In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path(r"E:\Projet_Alzheimer")
IRM = ROOT / "data" / "03_IRM_originales"

bilan = []

for numero in range(1, 23):
    dossier = IRM / f"IRM_359_LOT_{numero:03d}"

    nombre_dicom = 0
    taille_dicom = 0

    if dossier.is_dir():
        for fichier in dossier.rglob("*"):
            if fichier.is_file() and fichier.suffix.lower() == ".dcm":
                nombre_dicom += 1
                taille_dicom += fichier.stat().st_size

    bilan.append({
        "lot": f"{numero:03d}",
        "dossier_present": dossier.is_dir(),
        "fichiers_dcm": nombre_dicom,
        "taille_dcm_Go": round(taille_dicom / 1024**3, 3),
    })

inventaire = pd.DataFrame(bilan)
display(inventaire)

print("Total de fichiers DICOM :", inventaire["fichiers_dcm"].sum())
print(
    "Lots sans fichier .dcm détecté :",
    int((inventaire["fichiers_dcm"] == 0).sum()),
)

In [ ]:
from pathlib import Path
from collections import defaultdict
import re
import pandas as pd

ROOT = Path(r"E:\Projet_Alzheimer")
IRM = ROOT / "data" / "03_IRM_originales"
COHORTE = (
    ROOT / "data" / "05_Manifestes"
    / "cohorte_359_20261024_184212_495491"
)

# Identifiants attendus dans notre sélection.
manifeste = pd.read_csv(
    COHORTE / "series_IRM_candidates.csv",
    dtype="string",
)

ids_attendus = (
    manifeste["Image ID"]
    .str.strip()
    .str.replace(r"^[Ii]", "", regex=True)
)

if ids_attendus.isna().any() or not ids_attendus.str.fullmatch(r"\d+").all():
    raise ValueError("Identifiants invalides dans le manifeste.")

attendus = set(ids_attendus)

# Inventaire des fichiers réellement extraits.
comptages = defaultdict(int)
emplacements = defaultdict(set)
lots_par_image = defaultdict(set)
non_identifies = 0
vides = 0

for numero in range(1, 23):
    lot = f"IRM_359_LOT_{numero:03d}"
    dossier = IRM / lot

    for fichier in dossier.rglob("*"):
        if not fichier.is_file() or fichier.suffix.lower() != ".dcm":
            continue

        # Chercher l'identifiant dans le nom du fichier.
        correspondance = re.search(
            r"_I(\d+)(?=[_.]|$)", fichier.name, flags=re.IGNORECASE
        )

        # Sinon, chercher un dossier nommé Ixxxxx.
        if correspondance is None:
            for partie in reversed(fichier.relative_to(dossier).parts[:-1]):
                correspondance = re.fullmatch(
                    r"I(\d+)", partie, flags=re.IGNORECASE
                )
                if correspondance:
                    break

        if fichier.stat().st_size == 0:
            vides += 1

        if correspondance is None:
            non_identifies += 1
            continue

        identifiant = correspondance.group(1)
        comptages[identifiant] += 1
        emplacements[identifiant].add(str(fichier.parent))
        lots_par_image[identifiant].add(lot)

    print(f"Lot {numero:03d} vérifié")

presents = set(comptages)
manquants = attendus - presents
inattendus = presents - attendus

print("\nBILAN")
print("Séries attendues :", len(attendus))
print("Séries présentes :", len(presents))
print("Séries attendues retrouvées :", len(attendus & presents))
print("Séries manquantes :", len(manquants))
print("Séries inattendues :", len(inattendus))
print("Fichiers DICOM sans identifiant reconnu :", non_identifies)
print("Fichiers DICOM de taille nulle :", vides)
print(
    "Séries présentes dans plusieurs lots :",
    sum(len(lots) > 1 for lots in lots_par_image.values()),
)

# Sauvegarder un rapport local sans modifier les images.
rapport = pd.DataFrame([
    {
        "Image ID": identifiant,
        "attendue": identifiant in attendus,
        "nombre_fichiers_dcm": comptages[identifiant],
        "nombre_dossiers": len(emplacements[identifiant]),
        "lots": ";".join(sorted(lots_par_image[identifiant])),
    }
    for identifiant in sorted(attendus | presents, key=int)
])

horodatage = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
rapport.to_csv(
    COHORTE / f"verification_presence_IRM_{horodatage}.csv",
    index=False,
)

In [ ]:
%pip install pydicom

In [ ]:
from pathlib import Path
import pandas as pd
import pydicom

# Utilise les variables de la cellule précédente :
# attendus, emplacements, COHORTE

lignes = []

for numero, image_id in enumerate(sorted(attendus, key=int), start=1):
    ligne = {"Image ID": image_id}

    try:
        fichier = next(
            f
            for dossier in sorted(emplacements[image_id])
            for f in Path(dossier).iterdir()
            if f.is_file() and f.suffix.lower() == ".dcm"
        )

        ds = pydicom.dcmread(fichier, stop_before_pixels=True)

        ligne.update({
            "statut": "OK",
            "fichier_exemple": str(fichier),
            "patient_dicom": str(ds.get("PatientID", "")),
            "date_dicom": str(ds.get("StudyDate", "")),
            "modalite": str(ds.get("Modality", "")),
            "description": str(ds.get("SeriesDescription", "")),
            "series_uid": str(ds.get("SeriesInstanceUID", "")),
            "lignes_pixels": str(ds.get("Rows", "")),
            "colonnes_pixels": str(ds.get("Columns", "")),
            "espacement_pixels": str(ds.get("PixelSpacing", "")),
            "epaisseur_coupe": str(ds.get("SliceThickness", "")),
            "champ_magnetique": str(ds.get("MagneticFieldStrength", "")),
            "fabricant": str(ds.get("Manufacturer", "")),
        })

    except Exception as erreur:
        ligne.update({
            "statut": "ERREUR",
            "erreur": f"{type(erreur).__name__}: {erreur}",
        })

    lignes.append(ligne)

    if numero % 250 == 0:
        print(f"{numero} / {len(attendus)} séries examinées")

inventaire = pd.DataFrame(lignes)

horodatage = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
destination = COHORTE / f"inventaire_entetes_IRM_{horodatage}.csv"
inventaire.to_csv(destination, index=False)

print("\nBILAN DES EN-TÊTES")
print(inventaire["statut"].value_counts().to_string())

if "modalite" in inventaire.columns:
    print("\nModalités :")
    print(inventaire["modalite"].value_counts(dropna=False).to_string())

print("\nInventaire enregistré :", destination)

In [ ]:
from pathlib import Path
import pandas as pd

COHORTE = Path(
    r"E:\Projet_Alzheimer\data\05_Manifestes"
    r"\cohorte_359_20261024_184212_495491"
)

inventaire = pd.read_csv(
    COHORTE / "inventaire_entetes_IRM_20261026_221604_056668.csv",
    dtype="string"
)

selection = pd.read_csv(
    COHORTE / "series_IRM_candidates.csv",
    dtype="string"
)

# Harmoniser les identifiants des images
for table in (inventaire, selection):
    table["Image ID"] = (
        table["Image ID"].str.strip()
        .str.replace(r"^[Ii]", "", regex=True)
    )

controle = selection[
    ["Image ID", "PTID", "date_irm"]
].merge(
    inventaire,
    on="Image ID",
    how="outer",
    validate="one_to_one",
    indicator=True
)

# Extraire le code participant de l'en-tête
controle["participant_entete"] = controle["patient_dicom"].str.extract(
    r"(\d{3}_S_\d{4})", expand=False
)

controle["date_attendue"] = pd.to_datetime(
    controle["date_irm"], errors="coerce"
).dt.normalize()

controle["date_entete"] = pd.to_datetime(
    controle["date_dicom"],
    format="%Y%m%d",
    errors="coerce"
)

controle["participant_concordant"] = (
    controle["PTID"].str.strip()
    .eq(controle["participant_entete"])
    .fillna(False)
)

controle["date_concordante"] = (
    controle["date_attendue"].notna()
    & controle["date_entete"].notna()
    & controle["date_attendue"].eq(controle["date_entete"])
)

print("Séries comparées :", len(controle))
print("Présentes dans les deux fichiers :",
      controle["_merge"].eq("both").sum())
print("Participants concordants :",
      controle["participant_concordant"].sum())
print("Dates concordantes :", controle["date_concordante"].sum())
print("Identifiants participant non reconnus :",
      controle["participant_entete"].isna().sum())
print("Dates DICOM absentes ou non reconnues :",
      controle["date_entete"].isna().sum())

horodatage = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
controle.to_csv(
    COHORTE / f"controle_correspondance_IRM_{horodatage}.csv",
    index=False
)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"E:\Projet_Alzheimer")
COHORTE = (
    ROOT / "data" / "05_Manifestes"
    / "cohorte_359_20261024_184212_495491"
)

# 1. Recharger les visites retenues
visites = pd.read_csv(
    COHORTE / "visites_candidates.csv",
    dtype="string"
)

cibles = visites[["PTID", "date_pet"]].copy()
cibles["PTID"] = cibles["PTID"].str.strip()
cibles["date_pet"] = pd.to_datetime(
    cibles["date_pet"], errors="raise"
).dt.normalize()

if cibles.isna().any().any():
    raise ValueError("Une cible PET contient un participant ou une date absente.")

cibles = cibles.drop_duplicates()

# 2. Charger le catalogue PET
pet = pd.read_csv(
    ROOT / "data" / "02_Tables_originales"
    / "All_Subjects_Key_PET_23Oct2026.csv",
    dtype="string"
)

pet["PTID"] = pet["subject_id"].str.strip()
pet["date_pet"] = pd.to_datetime(
    pet["image_date"], errors="coerce"
).dt.normalize()

# 3. Garder le traceur FDG
traceur = pet["radiopharmaceutical"].str.strip().str.upper()
fdg = pet.loc[traceur.eq("18F-FDG").fillna(False)].copy()

# 4. Retrouver les séries aux dates sélectionnées
candidats = cibles.merge(
    fdg,
    on=["PTID", "date_pet"],
    how="inner",
    validate="one_to_many"
)

dates_retrouvees = candidats[["PTID", "date_pet"]].drop_duplicates()

verification = cibles.merge(
    dates_retrouvees,
    on=["PTID", "date_pet"],
    how="left",
    indicator=True
)

manquantes = verification.loc[
    verification["_merge"].eq("left_only"),
    ["PTID", "date_pet"]
]

# 5. Sauvegarder sans modifier les fichiers précédents
horodatage = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
DOSSIER_PET = COHORTE / f"collecte_PET_{horodatage}"
DOSSIER_PET.mkdir(parents=True, exist_ok=False)

candidats.to_csv(
    DOSSIER_PET / "series_PET_FDG_candidates.csv",
    index=False
)
manquantes.to_csv(
    DOSSIER_PET / "dates_PET_non_retrouvees.csv",
    index=False
)

print("Participants ciblés :", cibles["PTID"].nunique())
print("Dates PET ciblées :", len(cibles))
print("Dates PET retrouvées :", len(dates_retrouvees))
print("Dates PET non retrouvées :", len(manquantes))
print("Lignes PET candidates :", len(candidats))
print("Identifiants d'image uniques :", candidats["image_id"].nunique())

print("\nDescriptions des séries :")
print(
    candidats["pet_description"]
    .fillna("NON_RENSEIGNE")
    .value_counts()
    .to_string()
)

print("\nDossier enregistré :", DOSSIER_PET)

In [ ]:
from pathlib import Path
import pandas as pd
import pydicom
import numpy as np
import warnings

# 1. Charger l'inventaire des IRM
COHORTE = Path(
    r"E:\Projet_Alzheimer\data\05_Manifestes"
    r"\cohorte_359_20261024_184212_495491"
)

inventaire = pd.read_csv(
    COHORTE / "inventaire_entetes_IRM_20261026_221604_056668.csv",
    dtype="string"
).rename(columns={"Image ID": "image_id"})

colonnes_requises = {"image_id", "fichier_exemple"}

if not colonnes_requises.issubset(inventaire.columns):
    raise ValueError("L'inventaire ne contient pas les colonnes nécessaires.")

if inventaire[["image_id", "fichier_exemple"]].isna().any().any():
    raise ValueError("Identifiant ou chemin de fichier absent dans l'inventaire.")

if inventaire["image_id"].duplicated().any():
    raise ValueError("Identifiants d'image en double dans l'inventaire.")

# 2. Créer un nouveau dossier de résultats
horodatage = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
SORTIE = COHORTE / f"lecture_pixels_IRM_{horodatage}"
SORTIE.mkdir(parents=True, exist_ok=False)

bilans = []
anomalies = []

print("Séries à examiner :", len(inventaire))
print("Dossier des résultats :", SORTIE)

# 3. Examiner les fichiers de chaque série
for numero, (_, serie) in enumerate(inventaire.iterrows(), start=1):
    image_id = serie["image_id"]
    dossier = Path(serie["fichier_exemple"]).parent

    fichiers = []
    erreur_dossier = ""

    try:
        fichiers = sorted(
            fichier
            for fichier in dossier.iterdir()
            if fichier.is_file() and fichier.suffix.lower() == ".dcm"
        )
    except OSError as erreur:
        erreur_dossier = f"{type(erreur).__name__}: {erreur}"
        anomalies.append({
            "image_id": image_id,
            "fichier": str(dossier),
            "type": "erreur_dossier",
            "message": erreur_dossier
        })

    lus = 0
    erreurs = 0
    non_finis = 0
    nombre_avertissements = 0
    formes = set()

    for fichier in fichiers:
        ds = None
        pixels = None

        with warnings.catch_warnings(record=True) as alertes:
            warnings.simplefilter("always")

            try:
                ds = pydicom.dcmread(fichier)
                pixels = ds.pixel_array

                formes.add(str(pixels.shape))
                lus += 1

                if not np.isfinite(pixels).all():
                    non_finis += 1
                    anomalies.append({
                        "image_id": image_id,
                        "fichier": str(fichier),
                        "type": "valeurs_non_finies",
                        "message": "Présence de valeurs NaN ou infinies."
                    })

            except Exception as erreur:
                erreurs += 1
                anomalies.append({
                    "image_id": image_id,
                    "fichier": str(fichier),
                    "type": "erreur",
                    "message": f"{type(erreur).__name__}: {erreur}"
                })

            finally:
                del pixels, ds

            nombre_avertissements += len(alertes)

            for alerte in alertes:
                anomalies.append({
                    "image_id": image_id,
                    "fichier": str(fichier),
                    "type": "avertissement",
                    "message": str(alerte.message)
                })

    bilans.append({
        "image_id": image_id,
        "fichiers_trouves": len(fichiers),
        "fichiers_decodes": lus,
        "erreurs": erreurs,
        "erreur_dossier": erreur_dossier,
        "fichiers_valeurs_non_finies": non_finis,
        "nombre_avertissements": nombre_avertissements,
        "formes_pixels": ";".join(sorted(formes))
    })

    # 4. Sauvegarder régulièrement la progression
    if numero % 50 == 0 or numero == len(inventaire):
        pd.DataFrame(bilans).to_csv(
            SORTIE / "bilan_pixels.csv",
            index=False
        )

        pd.DataFrame(
            anomalies,
            columns=["image_id", "fichier", "type", "message"]
        ).to_csv(
            SORTIE / "anomalies.csv",
            index=False
        )

        print(f"{numero} / {len(inventaire)} séries examinées", flush=True)

# 5. Afficher le bilan final
bilan = pd.DataFrame(bilans)

print("\nBILAN DE LECTURE DES PIXELS IRM")
print("Séries examinées :", len(bilan))
print("Fichiers trouvés :", bilan["fichiers_trouves"].sum())
print("Fichiers décodés :", bilan["fichiers_decodes"].sum())
print("Erreurs de lecture/décodage :", bilan["erreurs"].sum())
print("Dossiers inaccessibles :", bilan["erreur_dossier"].ne("").sum())
print("Dossiers sans DICOM :", bilan["fichiers_trouves"].eq(0).sum())
print(
    "Fichiers avec valeurs non finies :",
    bilan["fichiers_valeurs_non_finies"].sum()
)
print(
    "Séries avec avertissements :",
    bilan["nombre_avertissements"].gt(0).sum()
)
print(
    "Total conforme à l'inventaire précédent (366973 fichiers) :",
    bilan["fichiers_trouves"].sum() == 366973
)

print("\nNombre de fichiers par série :")
print(
    bilan["fichiers_trouves"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nDimensions des coupes par série :")
print(bilan["formes_pixels"].value_counts().to_string())

print("\nRapports enregistrés :", SORTIE)